In [5]:
import os
import csv
import psycopg2

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [333]:
folder_path = os.getcwd()  # change if needed
folder_path

'/Users/alikhanzadi/Desktop/Learn/Projects/athl-data-platform/data'

In [106]:
import pandas as pd
import numpy as np
import uuid
import random
from datetime import datetime, timedelta

np.random.seed(42)


# ------------------------
# CONFIG
# ------------------------
NUM_USERS = 2000
NUM_ISSUERS = 200
NUM_TOKENS = 150
NUM_TRANSACTIONS = 50000

START_DATE = datetime(2025, 1, 1)

PRICE_INCREMENT = 0.000023
MAX_PRICE = 24.0

def generate_uuid():
    return str(uuid.uuid4())

def random_timestamp():
    return START_DATE + timedelta(days=random.randint(0, 365))

In [ ]:
# from io import StringIO

# from psycopg2 import sql

# # Get the connection string from the environment variable
# conn_string = os.getenv("DATABASE_URL")


# # def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
# #     """Bulk-load a DataFrame into Postgres via COPY.

# #     By default clears the table first with ``TRUNCATE ... RESTART IDENTITY`` so each
# #     run replaces rows (avoids duplicate key errors on reload).

# #     If other tables reference this one, plain ``TRUNCATE`` may fail; set
# #     ``truncate_cascade=True`` to truncate dependent tables too (destructive).

# #     Set ``clear_first=False`` to append without clearing.
# #     """
# #     buf = StringIO()
# #     df.to_csv(buf, index=False, header=True)
# #     buf.seek(0)
# #     with psycopg2.connect(conn_string) as conn:
# #         with conn.cursor() as cur:
# #             cur.execute(sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY CASCADE" ).format(sql.Identifier(table_name)))
# #             #         )
# #             # if clear_first:
# #             #     if truncate_cascade:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY CASCADE"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             #     else:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             copy_sql = sql.SQL(
# #                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
# #             ).format(sql.Identifier(table_name))
# #             cur.copy_expert(copy_sql, buf)
# #         conn.commit()
# #     print(f"Loaded {len(df)} rows into {table_name!r}")

# def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)

#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             # 1. Properly execute the TRUNCATE
#             if clear_first:
#                 # Add CASCADE if requested
#                 suffix = " CASCADE" if truncate_cascade else ""
#                 truncate_stmt = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
#                     sql.Identifier(table_name)
#                 )
#                 cur.execute(truncate_stmt) # This sends the command to the DB

#             # 2. Perform the COPY
#             copy_sql = sql.SQL(
#                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#             ).format(sql.Identifier(table_name))
#             cur.copy_expert(copy_sql, buf)
            
#         # 3. Commit the transaction (essential!)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")


In [12]:
import os
import psycopg2
from psycopg2 import sql
from io import StringIO
conn_string = os.getenv("DATABASE_URL")

def truncate_table(conn_string, table_name, cascade=False):
    """Forcefully clears the table and resets identities."""
    suffix = " CASCADE" if cascade else ""
    query = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
        sql.Identifier(table_name)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
        conn.commit()
    print(f"Table {table_name!r} truncated successfully.")

# def copy_data(conn_string, df, table_name):
#     """Loads DataFrame into the table via COPY."""
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)
    
#     copy_sql = sql.SQL(
#         "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#     ).format(sql.Identifier(table_name))
    
#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             cur.copy_expert(copy_sql, buf)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")

def copy_data(conn_string, df, table_name):
    buf = StringIO()
    df.to_csv(buf, index=False, header=True)
    buf.seek(0)
    
    # Identify the columns from the DataFrame
    columns = [sql.Identifier(col) for col in df.columns]
    
    # Format: COPY table_name (col1, col2, ...) FROM STDIN ...
    copy_sql = sql.SQL(
        "COPY {} ({}) FROM STDIN WITH (FORMAT csv, HEADER true)"
    ).format(
        sql.Identifier(table_name),
        sql.SQL(', ').join(columns)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(copy_sql, buf)
        conn.commit()
    print(f"Loaded {len(df)} rows into {table_name!r}")


In [ ]:
# ------------------------
# USERS
# ------------------------
users = []

for _ in range(NUM_USERS):
    user_id = generate_uuid()
    users.append({
        "user_id": user_id,
        "user_role": random.choice(["FAN", "ISSUER", "BOTH"]),
        "display_name": f"user_{random.randint(1000,9999)}",
        "username": f"user_{1000 + _}",
        # "username": f"user_{random.randint(10000,99999)}",
        "referral_id": None,
        "email": f"user{10000 + _}@test.com",
        # "email": f"user{random.randint(10000,99999)}@test.com",
        "email_verified": random.choice([True, False]),
        "email_verification_status": random.choice(["PENDING","VERIFIED"]),
        "email_verification_score": round(random.uniform(50,100),2),
        "country": random.choice(["US","CA","UK"]),
        "zip_code": str(random.randint(10000,99999)),
        "gender": random.choice(["Male","Female","Other"]),
        "time_zone": "UTC",
        # "login_methods_allowed": ["EMAIL_PASSWORD"],
        "password_hash": "hash",
        "passkey_public_key": None,
        "primary_social_platform": None,
        "primary_social_handle": None,
        "mfa_enabled": False,
        "account_status": "ACTIVE",
        "created_at": random_timestamp(),
        "updated_at": None,
        "last_login_at": None,
        "created_ip": "127.0.0.1",
        "created_user_agent": "generator",
        "metadata": None
    })
users_df = pd.DataFrame(users)
users_df.to_csv("./users.csv", index=False)
truncate_table(conn_string, 'users', cascade=True)
copy_data(conn_string, users_df,'users')


In [92]:
# ------------------------
# ISSUERS + VERIFICATION LOGIC
# ------------------------
issuers = []
identity = []
social = []

platforms = ["YOUTUBE", "INSTAGRAM", "X"]

issuer_users = users_df.sample(NUM_ISSUERS).reset_index(drop=True)

PASSED_BOTH = 150

# split remaining 50
remaining = NUM_ISSUERS - PASSED_BOTH

id_only_pass = 30
social_only_pass = 20
none_pass = remaining - id_only_pass - social_only_pass  # 10

for i, row in issuer_users.iterrows():
    issuer_id = generate_uuid()

    # ---------------- STATUS ASSIGNMENT ----------------
    if i < PASSED_BOTH:
        id_status = "PASSED"
        social_status = "PASSED"

    elif i < PASSED_BOTH + id_only_pass:
        id_status = "PASSED"
        social_status = random.choice(["FAILED", "PENDING"])

    elif i < PASSED_BOTH + id_only_pass + social_only_pass:
        id_status = random.choice(["FAILED", "PENDING"])
        social_status = "PASSED"

    else:
        id_status = random.choice(["FAILED", "PENDING"])
        social_status = random.choice(["FAILED", "PENDING"])

    # issuer status derived
    issuer_status = "PASSED" if (id_status == "PASSED" and social_status == "PASSED") else (
        "PENDING" if "PENDING" in [id_status, social_status] else "FAILED"
    )

    # ---------------- ISSUER ----------------
    issuers.append({
        "issuer_id": issuer_id,
        "user_id": row["user_id"],
        "issuer_type": random.choice(["ATHLETE", "CREATOR"]),
        "email": row["email"],
        "username": row["username"],
        "status": issuer_status,
        "level": random.choice(["YOUTH","COLLEGE","PRO"]),
        "country": row["country"],
        "region": "NA",
        "created_at": row["created_at"],
        "updated_at": None,
        "metadata": None
    })

    # ---------------- IDENTITY ----------------
    identity.append({
        "identity_check_id": generate_uuid(),
        "issuer_id": issuer_id,
        "provider": "Persona",
        "status": id_status,
        "level": "basic",
        "alias_confidence": round(random.uniform(70,100),2),
        "opted_in": True,
        "initiated_at": random_timestamp(),
        "completed_at": random_timestamp(),
        "failure_reason": None if id_status == "PASSED" else "check_failed",
        "raw_response": None
    })

    # ---------------- SOCIAL ----------------
    for p in random.sample(platforms, k=1):
        social.append({
            "social_verif_id": generate_uuid(),
            "issuer_id": issuer_id,
            "platform": p,
            "handle": f"{p.lower()}_{random.randint(1000,9999)}",
            "followers_count": random.randint(1000,1000000),
            "verified": True if social_status == "PASSED" else False,
            "initiated_at": random_timestamp(),
            "completed_at": random_timestamp(),
            "attempts": random.randint(1,3),
            "status": social_status,
            "metadata": None
        })

issuers_df = pd.DataFrame(issuers)
identity_df = pd.DataFrame(identity)
social_df = pd.DataFrame(social)

issuers_df.to_csv("./issuers.csv", index=False)
identity_df.to_csv("./identity_verification.csv", index=False)
social_df.to_csv("./social_verification.csv", index=False)

truncate_table(conn_string, 'issuers', cascade=True)
copy_data(conn_string, issuers_df,'issuers')

truncate_table(conn_string, 'identity_verification', cascade=True)
copy_data(conn_string, identity_df,'identity_verification')

truncate_table(conn_string, 'social_verification', cascade=True)
copy_data(conn_string, social_df,'social_verification')

Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'
Table 'identity_verification' truncated successfully.
Loaded 200 rows into 'identity_verification'
Table 'social_verification' truncated successfully.
Loaded 200 rows into 'social_verification'


In [107]:
issuers_df.groupby('status').count()

,issuer_id,user_id,issuer_type,email,username,level,country,region,created_at,updated_at,metadata
status,,,,,,,,,,,
FAILED,27,27,27,27,27,27,27,27,27,0,0
PASSED,150,150,150,150,150,150,150,150,150,0,0
PENDING,23,23,23,23,23,23,23,23,23,0,0


In [110]:
d = pd.merge(social_df, issuers_df[['issuer_id', 'status']], on='issuer_id', how='inner')
pd.cut(d[d['status_y']=='PASSED']['followers_count'], bins=[0, 100000, 500000, 1000000]).value_counts()

followers_count
(500000, 1000000]    76
(100000, 500000]     62
(0, 100000]          12
Name: count, dtype: int64

In [37]:
# # ------------------------
# # ISSUERS
# # ------------------------
# issuer_users = users_df.sample(NUM_ISSUERS)
# issuers = []

# for _, row in issuer_users.iterrows():
#     issuer_id = generate_uuid()
#     issuers.append({
#         "issuer_id": issuer_id,
#         "user_id": row["user_id"],
#         "issuer_type": random.choices(["ATHLETE","CREATOR"],weights=[8,1])[0],
#         "email": row["email"],
#         "username": row["username"],
#         "status" : random.choices(["PENDING","PASSED","FAILED"],weights=[2,7.5,0.5])[0],
#         "level": random.choice(["YOUTH","COLLEGE","PRO"]),
#         "country": row["country"],
#         "region": "NA",
#         "created_at": row["created_at"],
#         "updated_at": None,
#         "metadata": None
#     })
# issuers_df = pd.DataFrame(issuers)
# issuers_df.to_csv("./issuers.csv", index=False)
# truncate_table(conn_string, 'issuers', cascade=True)
# copy_data(conn_string, issuers_df,'issuers')



Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'


In [53]:
# ------------------------
# ATHLETE PROFILE
# ------------------------
athletes = []
for _, row in issuers_df.iterrows():
    if row["issuer_type"] == "ATHLETE":
        athletes.append({
            "issuer_id": row["issuer_id"],
            "user_id": row["user_id"],
            "sport": random.choice(["Basketball","Soccer","Football"]),
            "team": f"Team_{random.randint(1,20)}",
            "league": "Youth",
            "position_primary": "Forward",
            "position_secondary": None,
            "multi_sport": False,
            "biography": "bio",
            "profile_completion": random.randint(50,100),
            "created_at": random_timestamp(),
            "updated_at": None,
            "metadata": None
        })
athlete_df = pd.DataFrame(athletes)

athlete_df.to_csv("./athletes.csv", index=False)

truncate_table(conn_string, 'athlete_profile', cascade=True)
copy_data(conn_string, athlete_df,'athlete_profile')

Table 'athlete_profile' truncated successfully.
Loaded 200 rows into 'athlete_profile'


In [332]:
# ------------------------
# POST SIGNUP 
# ------------------------
post_signup = []

for _, row in issuers_df.iterrows():
    post_signup.append({
        "issuer_id": row["issuer_id"],
        "wallet_provisioned": random.choice([True, False]),
        "wallet_address": f"0x{uuid.uuid4().hex[:40]}",
        "wallet_email_sent": True,
        "dashboard_redirect": True,
        "oauth_verified_min2": random.choice([True, False]),
        "verified_at": random_timestamp(),
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

post_signup_df = pd.DataFrame(post_signup)

post_signup_df.to_csv("./issuer_post_signup.csv", index=False)

truncate_table(conn_string, 'issuer_post_signup', cascade=True)
copy_data(conn_string, post_signup_df,'issuer_post_signup')

Table 'issuer_post_signup' truncated successfully.
Loaded 200 rows into 'issuer_post_signup'


In [293]:
# ------------------------
# ISSUER PREFERENCES 
# ------------------------
preferences = []

for _, row in issuers_df.iterrows():
    preferences.append({
        "issuer_id": row["issuer_id"],
        "raise_target_usd": random.randint(10000,1000000), # not needed
        "token_supply_goal": random.randint(100000,1000000), # issuers with higher social followers should have a higher number (loosely)
        "enable_referrals": False,
        "notification_prefs": None,
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

preferences_df = pd.DataFrame(preferences)

preferences_df.to_csv("./issuer_preferences.csv", index=False)

truncate_table(conn_string, 'issuer_preferences', cascade=True)
copy_data(conn_string, preferences_df,'issuer_preferences')

Table 'issuer_preferences' truncated successfully.
Loaded 200 rows into 'issuer_preferences'


In [324]:
# ------------------------
# TOKENS  (ONLY VERIFIED ISSUERS)
# ------------------------
tokens = []
eligible_issuers = issuers_df[issuers_df["status"] == 'PASSED']

for _, row in eligible_issuers.sample(min(NUM_TOKENS, len(eligible_issuers))).reset_index(drop=True).iterrows():
    tokens.append({
        "token_id": len(tokens) + 1,
        "issuer_id": row["issuer_id"],
        "token_symbol": f"TOKEN_{random.randint(1000,9999)}",
        "initial_supply": random.randint(100000,1000000),
        "current_supply_minted": 0,
        # "numbers_sold": ##added later
        # "current_price": ##added later
        # "total_revenue_USD": ##added later
        # "bonding_curve_start_price": 1.000000,
        # "bonding_curve_end_price": 24.000000,
        "mint_timestamp": random_timestamp(),
        "paused_sales": False,
        "secondary_trading_enabled": False,
        "created_at": random_timestamp(),
        "updated_at": random_timestamp()
    })

tokens_df_temp = pd.DataFrame(tokens)

In [ ]:
# ------------------------
# TRANSACTIONS (PRIMARY ONLY)
# ------------------------
# transactions = []
# token_prices = {tid: 1.0 for tid in tokens_df["token_id"]}

# user_ids = users_df["user_id"].tolist()
# token_ids = tokens_df["token_id"].tolist()

# for i in range(NUM_TRANSACTIONS):
#     token_id = random.choice(token_ids)

#     current_price = token_prices[token_id]
#     new_price = min(MAX_PRICE, current_price + PRICE_INCREMENT)

#     token_prices[token_id] = new_price

#     quantity = random.randint(1, 20)

#     transactions.append({
#         "transaction_id": i + 1,
#         "token_id": token_id,
#         "buyer_id": random.choice(user_ids),
#         "seller_id": None,
#         "quantity": quantity,
#         #starting price
#         "price_per_token": format(new_price, ".6f"),# needs to be new price
#         "total_amount_usdc": format(quantity * new_price, ".6f"),
#         "transaction_type": "primary",
#         "status": "completed",
#         "timestamp": random_timestamp()
#     })

# transactions_df = pd.DataFrame(transactions)

# NEED TRANSACTION SIMULATION ENGINE

In [ ]:
# ------------------------
# TRANSACTIONS (PRIMARY ONLY)
# ------------------------
from decimal import Decimal, getcontext
getcontext().prec = 28

PRICE_INCREMENT = Decimal(0.000023)
START_PRICE = Decimal(1.0)
MAX_PRICE = Decimal(24.0)

# ---------------- BUILD ISSUER FOLLOWER MAP ----------------
# take max followers per issuer from social table
issuer_followers = (
    social_df.groupby("issuer_id")["followers_count"]
    .max()
    .to_dict()
)

# ---------------- HELPER ----------------
def total_cost(n):
    n = Decimal(n)
    first = START_PRICE
    last = START_PRICE + (n - 1) * PRICE_INCREMENT
    return (n / 2) * (first + last)

# ---------------- TRANSACTIONS ----------------
transactions = []
transaction_id = 1

user_ids = users_df["user_id"].tolist()
active_users = random.sample(user_ids, int(len(user_ids) * 0.65))

# build mapping once
issuer_to_user = dict(zip(issuers_df["issuer_id"], issuers_df["user_id"]))

for _, token in tokens_df_temp.iterrows():
    token_id = token["token_id"]
    issuer_id = token["issuer_id"]

    followers = issuer_followers.get(issuer_id, 0)

    # ---------------- RULE 1: NO SALES ----------------
    if followers < 100001:
        continue

    # ---------------- DEMAND FUNCTION ----------------
    # scale sales with followers
    max_tokens_to_sell = int(min(
        token["initial_supply"],
        followers * random.uniform(0.01, 0.05)
    ))

    tokens_sold = 0

    while tokens_sold < max_tokens_to_sell:

        # batch size (simulate transactions)
        batch = random.randint(1, 50)

        if tokens_sold + batch > max_tokens_to_sell:
            batch = max_tokens_to_sell - tokens_sold

        if batch <= 0:
            break

        # ---------------- PRICE CALC ----------------
        start_n = tokens_sold + 1
        end_n = tokens_sold + batch

        # price of next token
        current_price = min(
            MAX_PRICE,
            START_PRICE + (Decimal(start_n - 1) * PRICE_INCREMENT)
        )

        final_price = min(
            MAX_PRICE,
            START_PRICE + (Decimal(end_n - 1) * PRICE_INCREMENT)
        )

        # total cost using arithmetic series
        total_amount = total_cost(end_n) - total_cost(start_n - 1)

        transactions.append({
            "transaction_id": transaction_id,
            "token_id": token_id,
            "buyer_id": random.choice(user_ids),
            "seller_id": issuer_to_user[issuer_id],
            "quantity": batch,
            "starting_price": float(current_price),
            "ending_price": float(final_price),
            "total_amount_usdc": float(total_amount),
            "transaction_type": "primary",
            "swap_api_reference": None,
            "original_currency": "USD",
            "status": "completed",
            "reversal_reason": None,
            "timestamp": random_timestamp(),
            "on_chain_tx_hash": None,
            "merkle_proof_hash": None
        })

        tokens_sold += batch
        transaction_id += 1

transactions_df = pd.DataFrame(transactions)
transactions_df["total_amount_usdc"] = transactions_df["total_amount_usdc"].astype(float).round(6)

transactions_df.to_csv("./transactions.csv", index=False)

truncate_table(conn_string, 'transactions', cascade=True)
copy_data(conn_string, transactions_df,'transactions')

Table 'transactions' truncated successfully.
Loaded 91560 rows into 'transactions'


In [308]:
transactions_df['total_amount_usdc'].sum()

np.float64(2957248.9334670007)

In [309]:
# # ------------------------
# # ISSUER TOKEN NET CREDIT LEDGER
# # ------------------------
# for _, tx in transactions_df.iterrows():
#     credits.append({
#         "credit_id": generate_uuid(),   
#         "issuer_id": tx["seller_id"],
#         "transaction_id": tx["transaction_id"],  
#         "amount_usdc": tx["total_amount_usdc"]*0.8,
#         "created_at": tx["timestamp"]
#     })

# issuer_token_net_credit_ledger = pd.DataFrame(credits)
# issuer_token_net_credit_ledger

In [310]:
# ------------------------
# ISSUER DAILY REVENUE
# ------------------------
issuer_daily_revenue_df = (
    transactions_df
    .groupby(["seller_id", "timestamp"])
    .agg({
        "total_amount_usdc": "sum"
    })
    .reset_index()
)

issuer_daily_revenue_df["total_amount_usdc"] =0.8* issuer_daily_revenue_df["total_amount_usdc"].round(6)
issuer_daily_revenue_df = issuer_daily_revenue_df.rename(columns={'seller_id':'issuer_id','timestamp':'date'})

issuer_daily_revenue_df.to_csv("./issuer_daily_revenue.csv", index=False)

truncate_table(conn_string, 'issuer_daily_revenue', cascade=True)
copy_data(conn_string, issuer_daily_revenue_df,'issuer_daily_revenue')


Table 'issuer_daily_revenue' truncated successfully.
Loaded 37632 rows into 'issuer_daily_revenue'


In [311]:
# ------------------------
# USER TOKEN WALLET
# ------------------------
user_token_wallet_df = (
    transactions_df
    .groupby(["buyer_id", "token_id"])
    .agg({
        "quantity": "sum",
        "total_amount_usdc": "sum"
    })
    .reset_index()
    .rename(columns={
        "buyer_id": "user_id",
        "total_amount_usdc": "total_value"
    })
)

# user_token_wallet_df["total_value"] = user_token_wallet_df["total_value"]
# user_token_wallet_df

user_token_wallet_df.to_csv("./user_token_wallet.csv", index=False)

truncate_table(conn_string, 'user_token_wallet', cascade=True)
copy_data(conn_string, user_token_wallet_df,'user_token_wallet')



Table 'user_token_wallet' truncated successfully.
Loaded 74066 rows into 'user_token_wallet'


In [ ]:
# ------------------------
# USER WALLET
# ------------------------

wallets = []

user_token_totals_USDC = (
    user_token_wallet_df.groupby("user_id")["total_value"]
    .sum()
    .to_dict()
)

issuer_revenue = (
    transactions_df
    .groupby("seller_id")["total_amount_usdc"]
    .sum()
    .to_dict()
)


for user_id in users_df["user_id"]:

    # ETH balance (random)
    eth_balance = round(random.uniform(0, 5), 6)

    # USDC
    if user_id in issuer_revenue:
        # issuer → must be ≥ revenue
        revenue = issuer_revenue[user_id]
        usdc_balance = revenue + random.uniform(0, 500)
    else:
        usdc_balance = random.uniform(0, 500)

    # token holdings
    # tokens = user_token_balances.get(user_id, {})
    token_value = user_token_totals_USDC.get(user_id, 0)

    wallets.append({
        "wallet_id": generate_uuid(),
        "user_id": user_id,
        "eth_balance": format(eth_balance, ".6f"),
        "usdc_balance": format(usdc_balance, ".6f"),
        "total_token_value": token_value,    
        # "token_balances": tokens,   # dict {token_id: qty}
        "created_at": random_timestamp()
    })

wallets_df = pd.DataFrame(wallets)

wallets_df.to_csv("./user_wallet.csv", index=False)

truncate_table(conn_string, 'user_wallet', cascade=True)
copy_data(conn_string, wallets_df,'user_wallet')


Table 'user_wallet' truncated successfully.
Loaded 2000 rows into 'user_wallet'


In [313]:
wallets_df

,wallet_id,user_id,eth_balance,usdc_balance,total_token_value,created_at
0,67192a86-fbc6-443f-8fd3-ca5dbbaaa740,1587f1a9-aec1-43da-87c4-b635d5c4fda2,1.373100,50.850343,1539.493751,2025-10-28
1,9668959e-316b-4496-a544-418b0509f3f2,b2adc31a-4fac-4061-9fa8-4ffcc0d061a7,4.124547,43098.046055,1381.504593,2025-03-24
2,e527e2af-630f-48cb-af82-81937e12a551,c03c032c-7658-45b8-bf3a-c0d63d896724,3.249914,449.536448,1660.129725,2025-06-25
3,9632acd2-0818-437b-bb2c-7ace03bc6779,a4ef01bc-6631-40ab-a782-51a30fae0a80,2.656771,326.307556,1826.713645,2025-02-23
4,346753e5-6711-473e-858c-0cb333865014,be1f590f-bb48-4182-b060-f900fc87ab3e,0.175507,141.244218,1137.597772,2025-12-23
...,...,...,...,...,...,...
1995,4a7db08e-9a76-4554-a765-6e6656371f6e,ad28f898-06ce-4242-b1a2-a2051e77af76,4.780216,60.751795,1444.149813,2025-08-09
1996,7fc54fcf-5cc1-4aa7-9df5-95697b36e8ea,f7d7cc6a-3124-459c-928d-d244538012e3,3.690814,103.617491,1325.951776,2025-03-23
1997,0cdba0e0-2be4-4021-b406-00396b1c85f6,f72d68b7-e688-45a0-ba15-b979b82ea34b,3.955602,348.671018,1089.364817,2025-12-07
1998,93e787b0-5ac2-4d31-987c-d0b5f810b84c,48592b45-0680-4c60-97f8-c7fe0d659d85,2.159926,24.840265,1628.652484,2025-07-14


In [330]:
# ------------------------
# TOKENS FINAL
# ------------------------

token_stats = (
    transactions_df
    .groupby("token_id")
    .agg({
        "quantity": "sum",
        "total_amount_usdc": "sum",
        # "ending_price" :"max"
        
    })
    .reset_index()
)

token_stats["current_price"] = (
    1 + token_stats["quantity"] * 0.000023
).clip(upper=24)

# round
token_stats["current_price"] = token_stats["current_price"].round(6) 
token_stats["total_amount_usdc"] = token_stats["total_amount_usdc"].round(6)

# # merge into existing tokens_df
tokens_df = tokens_df_temp.merge(token_stats, on="token_id", how="left")

tokens_df["quantity"] = tokens_df["quantity"].fillna(0)
tokens_df["total_amount_usdc"] = tokens_df["total_amount_usdc"].fillna(0)
tokens_df["current_price"] = tokens_df["current_price"].fillna(1.0)
tokens_df.rename(columns={
    "quantity": "total_sold",
    "total_amount_usdc": "total_revenue"
}, inplace=True)

tokens_df.to_csv("./tokens.csv", index=False)

truncate_table(conn_string, 'tokens', cascade=True)
copy_data(conn_string, tokens_df,'tokens')

Table 'tokens' truncated successfully.
Loaded 150 rows into 'tokens'
